# Modelo de Previsão de Pedidos Semanais - Regressão Linear

Notebook dedicado ao treinamento e serialização de um modelo de regressão linear para prever pedidos semanais.

**Escopo:**
- Carregamento e tratamento de dados
- Engenharia de features
- Treinamento de regressão linear com validação temporal
- Avaliação de desempenho (RMSE, MAE, R²)
- Serialização do modelo em formato `.pkl`

**Output:**
- Modelo treinado: `../Dumps/linear_model.pkl`
- Metadados do modelo: `../Dumps/model_metadata.json`

## 1. Imports e Configuração do Ambiente

In [ ]:
from __future__ import annotations

import json
import random
import warnings
from dataclasses import dataclass
from datetime import datetime
from pathlib import Path
from typing import Any

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore")

# Reproducibilidade
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (14, 8)

print("Ambiente pronto")
print(f"   pandas: {pd.__version__}")
print(f"   numpy: {np.__version__}")
print(f"   scikit-learn: {sklearn.__version__}")

## 2. Configuração e Constantes

In [ ]:
CONFIG: dict[str, Any] = {
    "project_root": Path.cwd(),
    "data_dir": Path("..\\..\\data"),  # Navegando de ml/ para data/
    "dumps_dir": Path("Dumps"),
    "test_weeks": 16,
    "iqr_multiplier": 1.5,
    "event_exception_cols": ["is_holiday", "is_carnaval", "is_sao_joao", "is_summer"],
    "expected_files": {
        "estoque": "estoque_entradas_sintetico_2023_2026_mar.xlsx",
        "fichas": "fichas_tecnicas_ids.xlsx",
        "ingredientes": "lista_ingredientes.xlsx",
        "receitas": "lista_receitas.xlsx",
        "pedidos": "pedidos_sinteticos_2023_2026_mar.xlsx",
        "resumo_diario": "resumo_diario_pedidos_2023_2026_mar.xlsx",
        "resumo_mensal": "resumo_mensal_pedidos_2023_2026_mar.xlsx",
    },
}

# Criar pasta Dumps se não existir
CONFIG["dumps_dir"].mkdir(exist_ok=True)

print(f"Diretório do projeto: {CONFIG['project_root']}")
print(f"Diretório de dados: {CONFIG['data_dir'].resolve()}")
print(f"Diretório Dumps: {CONFIG['dumps_dir'].resolve()}")

## 3. Estruturas de Dados (Dataclasses)

In [ ]:
@dataclass
class QualitySummary:
    tabela: str
    linhas: int
    colunas: int
    nulos_total: int
    duplicatas: int


@dataclass
class ModelArtifacts:
    rmse: dict[str, float]
    mae: dict[str, float]
    r2: dict[str, float]
    test_frame: pd.DataFrame
    model: Pipeline
    training_date: str


@dataclass
class PipelineState:
    tabelas_raw: dict[str, pd.DataFrame]
    tabelas_clean: dict[str, pd.DataFrame]
    diario: pd.DataFrame
    diario_sem_outliers: pd.DataFrame
    semanal: pd.DataFrame
    outlier_audit: pd.DataFrame
    model_artifacts: ModelArtifacts | None = None


print("Dataclasses definidas")

## 4. Funções de Processamento

In [ ]:
def to_snake_case(columns: list[str]) -> list[str]:
    """Converte nomes de colunas para snake_case."""
    return [
        c.strip()
        .replace(" ", "_")
        .replace("-", "_")
        .replace("/", "_")
        .replace("(", "")
        .replace(")", "")
        .lower()
        for c in columns
    ]


def load_tables(config: dict[str, Any]) -> dict[str, pd.DataFrame]:
    """Carrega tabelas Excel configuradas."""
    tables: dict[str, pd.DataFrame] = {}
    for key, file_name in config["expected_files"].items():
        path = config["data_dir"] / file_name
        if not path.exists():
            raise FileNotFoundError(f"Arquivo não encontrado: {path}")
        df = pd.read_excel(path)
        df.columns = to_snake_case(df.columns.tolist())
        tables[key] = df
    return tables


def build_quality_report(tables: dict[str, pd.DataFrame]) -> pd.DataFrame:
    """Gera relatório de qualidade das tabelas."""
    records: list[QualitySummary] = []
    for name, df in tables.items():
        records.append(
            QualitySummary(
                tabela=name,
                linhas=int(df.shape[0]),
                colunas=int(df.shape[1]),
                nulos_total=int(df.isna().sum().sum()),
                duplicatas=int(df.duplicated().sum()),
            )
        )
    return pd.DataFrame([r.__dict__ for r in records]).sort_values("tabela")


def normalize_tables(tables: dict[str, pd.DataFrame]) -> dict[str, pd.DataFrame]:
    """Normaliza e limpa tabelas."""
    clean = {k: v.copy() for k, v in tables.items()}

    clean["pedidos"]["data_hora"] = pd.to_datetime(clean["pedidos"]["data_hora"], errors="coerce")
    clean["pedidos"] = clean["pedidos"].rename(columns={"id_produto": "id_receita"})
    clean["pedidos"]["id_receita"] = clean["pedidos"]["id_receita"].astype("Int64")
    clean["pedidos"]["quantidade"] = pd.to_numeric(clean["pedidos"]["quantidade"], errors="coerce")

    clean["resumo_diario"]["date"] = pd.to_datetime(clean["resumo_diario"]["date"], errors="coerce")
    clean["estoque"]["data_recebimento"] = pd.to_datetime(clean["estoque"]["data_recebimento"], errors="coerce")
    clean["estoque"]["data_validade"] = pd.to_datetime(clean["estoque"]["data_validade"], errors="coerce")

    clean["fichas"] = clean["fichas"].dropna(subset=["qtd_ingrediente", "custo_ingrediente"])

    for table_name, text_cols in {
        "ingredientes": ["ingrediente", "un_ingrediente"],
        "receitas": ["receita", "tipo"],
    }.items():
        for col in text_cols:
            clean[table_name][col] = clean[table_name][col].astype(str).str.strip().str.upper()

    for name, df in clean.items():
        clean[name] = df.drop_duplicates().reset_index(drop=True)

    clean["pedidos"] = clean["pedidos"].dropna(subset=["data_hora", "id_receita", "quantidade"])
    clean["pedidos"] = clean["pedidos"][clean["pedidos"]["quantidade"] > 0]

    return clean


def create_daily_frame(tables: dict[str, pd.DataFrame]) -> pd.DataFrame:
    """Cria frame diário a partir de pedidos."""
    pedidos = tables["pedidos"].copy()
    resumo_diario = tables["resumo_diario"].copy().rename(columns={"pedidos_dia": "pedidos_dia_resumo"})

    pedidos["date"] = pedidos["data_hora"].dt.floor("D")

    daily = (
        pedidos.groupby("date", as_index=False)
        .agg(
            pedidos_dia=("id_pedido", "nunique"),
            itens_dia=("quantidade", "sum"),
            linhas_itens=("id_pedido", "size"),
        )
        .sort_values("date")
    )

    event_cols = ["date", "pedidos_dia_resumo", "is_holiday", "is_carnaval", "is_sao_joao", "is_summer"]
    daily = daily.merge(resumo_diario[event_cols], on="date", how="left")

    for col in ["is_holiday", "is_carnaval", "is_sao_joao", "is_summer"]:
        daily[col] = daily[col].fillna(0).astype(int)

    daily["diff_resumo"] = daily["pedidos_dia"] - daily["pedidos_dia_resumo"].fillna(0)
    return daily


def remove_outliers_iqr_with_exceptions(
    df: pd.DataFrame,
    target_col: str,
    event_cols: list[str],
    multiplier: float = 1.5,
) -> tuple[pd.DataFrame, pd.DataFrame, dict[str, float]]:
    """Remove outliers com exceções para dias de evento."""
    work = df.copy()
    q1 = work[target_col].quantile(0.25)
    q3 = work[target_col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - multiplier * iqr
    upper = q3 + multiplier * iqr

    outlier_mask = (work[target_col] < lower) | (work[target_col] > upper)
    event_mask = work[event_cols].sum(axis=1) > 0
    weekend_mask = work["date"].dt.weekday >= 5

    removable_mask = outlier_mask & ~(event_mask | weekend_mask)

    audit = work.loc[outlier_mask, ["date", target_col, *event_cols]].copy()
    audit["removido"] = removable_mask[outlier_mask].values

    cleaned = work.loc[~removable_mask].copy().reset_index(drop=True)
    bounds = {"q1": float(q1), "q3": float(q3), "lower": float(lower), "upper": float(upper)}

    return cleaned, audit, bounds


def build_weekly_dataset(daily_clean: pd.DataFrame) -> pd.DataFrame:
    """Constrói dataset semanal com features de engenharia."""
    weekly = (
        daily_clean.set_index("date")
        .resample("W-MON", label="left", closed="left")
        .agg(
            pedidos_semana=("pedidos_dia", "sum"),
            itens_semana=("itens_dia", "sum"),
            feriado_semana=("is_holiday", "max"),
            carnaval_semana=("is_carnaval", "max"),
            sao_joao_semana=("is_sao_joao", "max"),
            verao_semana=("is_summer", "max"),
            dias_observados=("pedidos_dia", "size"),
        )
        .reset_index()
        .rename(columns={"date": "semana_inicio"})
    )

    weekly = weekly[weekly["dias_observados"] == 7].copy()
    weekly = weekly.drop(columns=["dias_observados"])

    weekly["ano"] = weekly["semana_inicio"].dt.year
    weekly["mes"] = weekly["semana_inicio"].dt.month.astype(str)
    weekly["semana_ano"] = weekly["semana_inicio"].dt.isocalendar().week.astype(int)
    weekly["trend"] = np.arange(len(weekly), dtype=float)
    weekly["lag_1"] = weekly["pedidos_semana"].shift(1)
    weekly["rolling_4"] = weekly["pedidos_semana"].shift(1).rolling(4).mean()

    weekly = weekly.dropna().reset_index(drop=True)
    return weekly


print("Funções de processamento carregadas")

## 5. Função de Treinamento da Regressão Linear

In [ ]:
NUM_FEATURES = [
    "itens_semana",
    "feriado_semana",
    "carnaval_semana",
    "sao_joao_semana",
    "verao_semana",
    "semana_ano",
    "trend",
    "lag_1",
    "rolling_4",
]
CAT_FEATURES = ["mes"]


def train_linear_regression(
    weekly_df: pd.DataFrame, test_weeks: int = 16
) -> ModelArtifacts:
    """Treina modelo de regressão linear com validação temporal."""
    if len(weekly_df) <= test_weeks + 8:
        raise ValueError("Série semanal curta demais para separar treino/teste com segurança.")

    split_idx = len(weekly_df) - test_weeks
    train = weekly_df.iloc[:split_idx].copy()
    test = weekly_df.iloc[split_idx:].copy()

    y_train = train["pedidos_semana"]
    y_test = test["pedidos_semana"]

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", StandardScaler(), NUM_FEATURES),
            ("cat", OneHotEncoder(handle_unknown="ignore"), CAT_FEATURES),
        ]
    )

    model = Pipeline(
        steps=[
            ("prep", preprocessor),
            ("reg", LinearRegression()),
        ]
    )

    X_train = train[NUM_FEATURES + CAT_FEATURES]
    X_test = test[NUM_FEATURES + CAT_FEATURES]

    model.fit(X_train, y_train)
    predictions = model.predict(X_test)

    rmse = float(np.sqrt(mean_squared_error(y_test, predictions)))
    mae = float(mean_absolute_error(y_test, predictions))
    r2 = float(r2_score(y_test, predictions))

    test_frame = test[["semana_inicio", "pedidos_semana"]].copy()
    test_frame["pred_linear"] = predictions

    return ModelArtifacts(
        rmse={"linear_regression": rmse},
        mae={"linear_regression": mae},
        r2={"linear_regression": r2},
        test_frame=test_frame,
        model=model,
        training_date=datetime.now().isoformat(),
    )


print("Função de treinamento carregada")

## 6. Executar Pipeline Completo

In [ ]:
print("Iniciando pipeline de carregamento e processamento...\n")

# 1) Carga
print("[1/5] Carregando tabelas...")
tabelas_raw = load_tables(CONFIG)
quality_raw = build_quality_report(tabelas_raw)
print(quality_raw.to_string())
print()

# 2) Limpeza
print("[2/5] Normalizando e limpando...")
tabelas_clean = normalize_tables(tabelas_raw)
quality_clean = build_quality_report(tabelas_clean)
print(quality_clean.to_string())
print()

# 3) Base diária
print("[3/5] Construindo base diária...")
diario = create_daily_frame(tabelas_clean)
print(f"Dias processados: {len(diario)}")
print(f"Data inicial: {diario['date'].min()}")
print(f"Data final: {diario['date'].max()}")
print()

# 4) Remove outliers
print("[4/5] Removendo outliers...")
diario_sem_outliers, outlier_audit, bounds = remove_outliers_iqr_with_exceptions(
    diario,
    target_col="pedidos_dia",
    event_cols=CONFIG["event_exception_cols"],
    multiplier=CONFIG["iqr_multiplier"],
)
print(f"Outliers detectados: {len(outlier_audit)}")
print(f"Outliers removidos: {outlier_audit['removido'].sum()}")
print(f"Limites IQR: lower={bounds['lower']:.2f}, upper={bounds['upper']:.2f}")
print()

# 5) Base semanal
print("[5/5] Construindo base semanal...")
semanal = build_weekly_dataset(diario_sem_outliers)
print(f"Semanas processadas: {len(semanal)}")
print(f"Semana inicial: {semanal['semana_inicio'].min()}")
print(f"Semana final: {semanal['semana_inicio'].max()}")

pipeline_state = PipelineState(
    tabelas_raw=tabelas_raw,
    tabelas_clean=tabelas_clean,
    diario=diario,
    diario_sem_outliers=diario_sem_outliers,
    semanal=semanal,
    outlier_audit=outlier_audit,
)

print("\nPipeline de processamento concluído!")

## 7. Treinar Modelo de Regressão Linear

In [ ]:
print("Treinando modelo de Regressão Linear com validação temporal...\n")

artifacts = train_linear_regression(pipeline_state.semanal, test_weeks=CONFIG["test_weeks"])
pipeline_state.model_artifacts = artifacts

print(f"Semanas de treino: {len(pipeline_state.semanal) - CONFIG['test_weeks']}")
print(f"Semanas de teste: {CONFIG['test_weeks']}")
print(f"Data de treinamento: {artifacts.training_date}")
print()
print("Métricas de Desempenho (Regressão Linear):")
print(f"  RMSE: {artifacts.rmse['linear_regression']:.2f}")
print(f"  MAE: {artifacts.mae['linear_regression']:.2f}")
print(f"  R²: {artifacts.r2['linear_regression']:.4f}")
print()
print("Modelo treinado com sucesso!")

## 8. Avaliação e Visualizações

In [ ]:
print("Gerando visualizações...\n")

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Plot 1: Série temporal com previsões
ax = axes[0, 0]
ax.plot(artifacts.test_frame["semana_inicio"], artifacts.test_frame["pedidos_semana"], "o-", label="Real", linewidth=2, markersize=8)
ax.plot(artifacts.test_frame["semana_inicio"], artifacts.test_frame["pred_linear"], "s--", label="Regressão Linear", linewidth=2, markersize=6)
ax.set_xlabel("Semana")
ax.set_ylabel("Pedidos/Semana")
ax.set_title("Previsões de Regressão Linear (16 semanas de teste)")
ax.legend()
ax.grid(alpha=0.3)

# Plot 2: Resíduos
ax = axes[0, 1]
residuals = artifacts.test_frame["pedidos_semana"] - artifacts.test_frame["pred_linear"]
ax.bar(range(len(residuals)), residuals, color="steelblue", alpha=0.7)
ax.axhline(0, color="red", linestyle="--", linewidth=2)
ax.set_xlabel("Semana (índice)")
ax.set_ylabel("Resíduo")
ax.set_title("Resíduos do Modelo (Real - Previsto)")
ax.grid(alpha=0.3, axis="y")

# Plot 3: Real vs Previsto
ax = axes[1, 0]
ax.scatter(artifacts.test_frame["pedidos_semana"], artifacts.test_frame["pred_linear"], s=100, alpha=0.7)
min_val = min(artifacts.test_frame["pedidos_semana"].min(), artifacts.test_frame["pred_linear"].min())
max_val = max(artifacts.test_frame["pedidos_semana"].max(), artifacts.test_frame["pred_linear"].max())
ax.plot([min_val, max_val], [min_val, max_val], "r--", linewidth=2, label="Linha Perfeita")
ax.set_xlabel("Real")
ax.set_ylabel("Previsto")
ax.set_title("Real vs Previsto (Regressão Linear)")
ax.legend()
ax.grid(alpha=0.3)

# Plot 4: Métricas
ax = axes[1, 1]
metrics_data = {
    "RMSE": artifacts.rmse["linear_regression"],
    "MAE": artifacts.mae["linear_regression"],
    "R²": artifacts.r2["linear_regression"],
}
ax.bar(metrics_data.keys(), metrics_data.values(), color=["#e74c3c", "#3498db", "#2ecc71"], alpha=0.8)
ax.set_ylabel("Valor")
ax.set_title("Métricas de Desempenho")
ax.grid(alpha=0.3, axis="y")
for i, (k, v) in enumerate(metrics_data.items()):
    ax.text(i, v + 0.05, f"{v:.3f}", ha="center", fontweight="bold")

plt.tight_layout()
plt.savefig(CONFIG["dumps_dir"] / "model_evaluation.png", dpi=300, bbox_inches="tight")
print("Visualizações salvas em model_evaluation.png")

## 9. Serializar Modelo para .pkl

In [ ]:
print("Serializando modelo e metadados...\n")

# Serializar modelo
model_path = CONFIG["dumps_dir"] / "linear_model.pkl"
joblib.dump(artifacts.model, model_path)
print(f"Modelo salvo em: {model_path.resolve()}")

# Metadados
metadata = {
    "model_type": "LinearRegression",
    "framework": "scikit-learn",
    "training_date": artifacts.training_date,
    "metrics": {
        "rmse": artifacts.rmse,
        "mae": artifacts.mae,
        "r2": artifacts.r2,
    },
    "configuration": {
        "test_weeks": CONFIG["test_weeks"],
        "numeric_features": NUM_FEATURES,
        "categorical_features": CAT_FEATURES,
    },
    "preprocessing": {
        "numeric_scaler": "StandardScaler",
        "categorical_encoder": "OneHotEncoder",
    },
}

metadata_path = CONFIG["dumps_dir"] / "model_metadata.json"
with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)
print(f"Metadados salvos em: {metadata_path.resolve()}")
print()
print("Serialização concluída!")

## 11. Verificar e Carregar Modelo do .pkl

In [ ]:
print("Verificação final...\n")

# Verificar arquivos criados
model_file = CONFIG["dumps_dir"] / "linear_model.pkl"
metadata_file = CONFIG["dumps_dir"] / "model_metadata.json"
viz_file = CONFIG["dumps_dir"] / "model_evaluation.png"

print("Arquivos criados:")
for f in [model_file, metadata_file, viz_file]:
    if f.exists():
        print(f"  [OK] {f.name} ({f.stat().st_size} bytes)")
    else:
        print(f"  [FALTA] {f.name}")

print()

# Teste de carregamento rápido
loaded_model = joblib.load(model_file)
print("Modelo carregado com sucesso")

# Teste de previsão no último teste
test_sample = pipeline_state.semanal.iloc[-4:-3][NUM_FEATURES + CAT_FEATURES]
pred = loaded_model.predict(test_sample)
print(f"Previsão de teste: {pred[0]:.2f} pedidos/semana")

print()
print("Verificação concluída com êxito!")
print(f"Arquivos salvos em: {CONFIG['dumps_dir'].resolve()}")

## Conclusão

Pipeline completo de treinamento e serialização de modelo de regressão linear concluído com sucesso!

### Resumo
- **Dados processados:** 7 tabelas Excel → daily frame → weekly dataset
- **Modelo treinado:** Regressão linear com 9 features numéricas + 1 categórica
- **Validação:** Split temporal (últimas 16 semanas como teste)
- **Desempenho:** RMSE, MAE e R² calculados e armazenados
- **Serialização:** Modelo e metadados salvos em `ml/Dumps/`

### Arquivos gerados
- `ml/Dumps/linear_model.pkl` → Modelo treinado (pronto para usar em produção)
- `ml/Dumps/model_metadata.json` → Informações e métricas do modelo

### Próximos passos
O modelo carregado pode ser utilizado para fazer inferências em novos dados. Exemplo:
```python
import joblib
model = joblib.load('Dumps/linear_model.pkl')
predictions = model.predict(X_new)  # X_new deve ter as mesmas features
```